In [ ]:
from pathlib import Path
import re
import warnings

import matplotlib

matplotlib.use("Agg")
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import numpy as np
import pandas as pd


# ── USER SETTINGS ──────────────────────────────────────────────────────────────
CSV_PATH = Path("./master.csv")
SUMMARY_DIR = Path("./ecDNA_summary")
OUT_PREFIX = Path("./neuroblastoma_heatmap")
OUTPUT_DPI = 900

# The qualitative palette is fixed across the figure. Amplicon IDs are local to
# each sample; equal colors therefore mean equal amplicon ID only within a column.
AMP_COLORS = {
    1: "#0072B2",  # blue
    2: "#E69F00",  # orange
    3: "#009E73",  # green
    4: "#D55E00",  # vermilion
    5: "#CC79A7",  # purple/pink
    6: "#56B4E9",  # sky blue
    7: "#F0E442",  # yellow
    8: "#8C564B",  # brown
    9: "#7F7F7F",  # gray
    10: "#17BECF", # cyan
}
C_EC_UNRESOLVED = "#6F4E7C"

# Publication-size typography. PDF text remains editable because TrueType fonts
# are embedded; the PNG and all rasterized artists are exported at 900 dpi.
FONT_SECTION = 9.0
FONT_ROW = 8.2
FONT_SAMPLE = 7.5
FONT_LEGEND = 8.0
FONT_CBAR = 7.5
plt.rcParams.update({
    "font.size": FONT_ROW,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.6,
})


# ── LOAD & PREP CSV ───────────────────────────────────────────────────────────
df_raw = pd.read_csv(CSV_PATH)
df_raw = df_raw.loc[:, ~df_raw.columns.str.startswith("Unnamed")]
df_raw = df_raw.rename(columns={
    "Sample":                     "sample",
    "MYCN_status":                "MYCN_status",
    "oncogene_on_ecDNA":          "ecDNA_genes",
    "ecDNA_counts":               "ecDNA_counts",
    "Chromothripsis_status":      "Chrom_status",
    "Chromothripsis_in_amplicon": "Chrom_in_amp",
    "MYCN_CN":                    "MYCN_CN",
    "Chromothripsis_counts":      "Chrom_counts",
    "SV_burden":                  "SV_burden",
    "MYCN_FPKM":                  "MYCN_FPKM",
    "CHEK1_FPKM":                 "CHEK1_FPKM",
})

required_cols = {
    "sample", "MYCN_status", "ecDNA_genes", "ecDNA_counts", "Chrom_status",
    "MYCN_CN", "MYCN_FPKM", "CHEK1_FPKM",
}
missing_cols = sorted(required_cols - set(df_raw.columns))
if missing_cols:
    raise ValueError(f"Missing required master.csv columns: {missing_cols}")
if df_raw["sample"].duplicated().any():
    dupes = df_raw.loc[df_raw["sample"].duplicated(keep=False), "sample"].tolist()
    raise ValueError(f"Sample names must be unique; duplicates: {dupes}")

df_raw = df_raw.set_index("sample")
df_raw["ecDNA_genes"] = df_raw["ecDNA_genes"].fillna("")
df = df_raw.sort_values(["MYCN_status", "MYCN_CN"], ascending=[False, False])
samples = df.index.tolist()
N = len(samples)


# ── READ AMPLICON-SPECIFIC ONCOGENES ──────────────────────────────────────────
ONCOGENE_RE = re.compile(
    r"^\[amplicon(?P<amp>\d+)\]\s+OncogenesAmplified\s*=\s*(?P<genes>.*)$",
    flags=re.IGNORECASE,
)
COUNT_RE = re.compile(r"^#Amplicons\s*=\s*(\d+)\s*$", flags=re.IGNORECASE)


def csv_genes(value):
    """Return de-duplicated comma-separated genes while preserving their order."""
    return list(dict.fromkeys(
        token.strip() for token in str(value).split(",") if token.strip()
    ))


def parse_summary(path):
    """Return ({amplicon_id: [genes]}, declared_amplicon_count)."""
    amp_to_genes = {}
    declared_count = None
    with path.open(encoding="utf-8", errors="replace") as handle:
        for raw_line in handle:
            line = raw_line.strip()
            count_match = COUNT_RE.match(line)
            if count_match:
                declared_count = int(count_match.group(1))
                continue
            match = ONCOGENE_RE.match(line)
            if not match:
                continue
            amp_id = int(match.group("amp"))
            genes = csv_genes(match.group("genes").rstrip(","))
            amp_to_genes.setdefault(amp_id, [])
            for gene in genes:
                if gene not in amp_to_genes[amp_id]:
                    amp_to_genes[amp_id].append(gene)
    return dict(sorted(amp_to_genes.items())), declared_count


# sample_gene_amplicons[sample][gene] -> tuple of one or more amplicon IDs.
# None means the gene is in master.csv but its amplicon could not be resolved.
sample_gene_amplicons = {sample: {} for sample in samples}
used_amp_ids = set()

if not SUMMARY_DIR.is_dir():
    warnings.warn(
        f"Summary directory not found: {SUMMARY_DIR}. "
        "ecDNA genes will be plotted as unresolved (?)."
    )

for sample in samples:
    expected_count = pd.to_numeric(df.loc[sample, "ecDNA_counts"], errors="coerce")
    expected_count = int(expected_count) if pd.notna(expected_count) else 0
    summary_path = SUMMARY_DIR / f"{sample}_summary.txt"
    csv_gene_list = csv_genes(df.loc[sample, "ecDNA_genes"])

    if summary_path.is_file():
        amp_to_genes, declared_count = parse_summary(summary_path)
        if declared_count is not None and declared_count != expected_count:
            warnings.warn(
                f"{sample}: master.csv ecDNA_counts={expected_count}, but "
                f"{summary_path.name} declares #Amplicons={declared_count}."
            )
        for amp_id, genes in amp_to_genes.items():
            used_amp_ids.add(amp_id)
            for gene in genes:
                previous = sample_gene_amplicons[sample].get(gene, ())
                sample_gene_amplicons[sample][gene] = tuple(sorted(set(previous) | {amp_id}))

        unresolved = [g for g in csv_gene_list if g not in sample_gene_amplicons[sample]]
        if unresolved:
            warnings.warn(
                f"{sample}: gene(s) listed in master.csv but not assigned to an "
                f"amplicon in {summary_path.name}: {', '.join(unresolved)}. "
                "They will be marked '?'."
            )
            for gene in unresolved:
                sample_gene_amplicons[sample][gene] = None
    else:
        if expected_count > 0:
            warnings.warn(
                f"{sample}: missing {summary_path}; CSV ecDNA genes will be marked '?'."
            )
        for gene in csv_gene_list:
            sample_gene_amplicons[sample][gene] = None


# Preserve the original CSV row order, then append any oncogenes found only in
# the summary files. This prevents summary-only annotations from disappearing.
all_ec_genes = []
for sample in samples:
    for gene in csv_genes(df.loc[sample, "ecDNA_genes"]):
        if gene not in all_ec_genes:
            all_ec_genes.append(gene)
for sample in samples:
    for gene in sample_gene_amplicons[sample]:
        if gene not in all_ec_genes:
            all_ec_genes.append(gene)


def amp_color(amp_id):
    if amp_id in AMP_COLORS:
        return AMP_COLORS[amp_id]
    # Deterministic extension for unusually high amplicon IDs.
    return matplotlib.colors.to_hex(plt.get_cmap("tab20")((amp_id - 1) % 20))


# Any extra columns are treated as mutation tracks.
core_cols = {
    "MYCN_status", "ecDNA_genes", "ecDNA_counts", "Chrom_status", "Chrom_in_amp",
    "MYCN_CN", "Chrom_counts", "SV_burden", "MYCN_FPKM", "CHEK1_FPKM",
}
mutation_genes = [column for column in df.columns if column not in core_cols]
has_mutations = bool(mutation_genes)


# ── COLORS AND ROWS ────────────────────────────────────────────────────────────
C_AMP = "#C0392B"
C_WT = "#27AE60"
C_OFF = "#EBEBEB"
C_CHR = "#1A5276"
C_MUT = "#3A86C8"

SECTIONS = [
    ("MYCN Amplification", [("", "MYCN_status", "binary_amp")]),
    ("ecDNA Counts", [("", "ecDNA_counts", "grad_purple")]),
    ("Chromothripsis", [("Status", "Chrom_status", "binary_blue")]),
    ("MYCN Status", [
        ("MYCN CN", "MYCN_CN", "grad_ylorrd"),
        ("MYCN FPKM", "MYCN_FPKM", "grad_mycnfpkm"),
    ]),
    ("Oncogenes on ecDNA", [(gene, gene, "ec_gene") for gene in all_ec_genes]),
    ("CHEK1 Expression", [("CHEK1 FPKM", "CHEK1_FPKM", "grad_chek1")]),
]
if has_mutations:
    SECTIONS.append(("Driver Mutations", [(g, g, "mutation") for g in mutation_genes]))


# ── LAYOUT ────────────────────────────────────────────────────────────────────
LABEL_W = 1.45
DATA_W = max(8.8, 0.43 * N)
CBAR_W = 0.20
CBAR_PAD = 0.34
RIGHT_PAD = 0.90
ROW_H = 0.25
TITLE_H = 0.30
GAP_H = 0.08
XLABEL_H = 0.88
BOTTOM_LEGEND_H = 1.05  # reserved for two compact legends below the plot
TOP_PAD_H = 0.12

rows_spec = []
for section_index, (section_title, section_rows) in enumerate(SECTIONS):
    rows_spec.append(("title", section_index, section_title, None, None))
    for row_label, key, row_type in section_rows:
        rows_spec.append(("data", section_index, row_label, key, row_type))

row_heights = [
    TITLE_H + (GAP_H if row[0] == "title" and row[1] > 0 else 0.0)
    if row[0] == "title" else ROW_H
    for row in rows_spec
]
row_heights.append(XLABEL_H)

plot_h = sum(row_heights)
total_h = plot_h + BOTTOM_LEGEND_H + TOP_PAD_H
total_w = LABEL_W + DATA_W + CBAR_PAD + CBAR_W + RIGHT_PAD
fig = plt.figure(figsize=(total_w, total_h), facecolor="white")

LABEL_FRAC = LABEL_W / (LABEL_W + DATA_W)
DATA_FRAC = DATA_W / (LABEL_W + DATA_W)
GS_RIGHT = (LABEL_W + DATA_W) / total_w
CBAR_X = (LABEL_W + DATA_W + CBAR_PAD) / total_w
CBAR_FW = CBAR_W / total_w
BOTTOM_FRAC = (BOTTOM_LEGEND_H + 0.03) / total_h
TOP_FRAC = 1.0 - TOP_PAD_H / total_h

nrows = len(row_heights)
gs = fig.add_gridspec(
    nrows, 1, height_ratios=row_heights, hspace=0.0,
    left=0.0, right=GS_RIGHT, bottom=BOTTOM_FRAC, top=TOP_FRAC,
)


# ── DRAWING HELPERS ───────────────────────────────────────────────────────────
def hide(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)


def render_title(ax, text):
    hide(ax)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.text(
        LABEL_FRAC + 0.005, 0.15, text, fontsize=FONT_SECTION,
        fontweight="bold", va="bottom", ha="left",
        transform=ax.transAxes, color="#1A1A1A",
    )
    ax.axhline(
        0.08, color="#CCCCCC", lw=0.7,
        xmin=LABEL_FRAC, xmax=LABEL_FRAC + DATA_FRAC,
    )


def render_row_label(ax, row_label, italic=False):
    ax.text(
        LABEL_FRAC - 0.008, 0.5, row_label, fontsize=FONT_ROW,
        va="center", ha="right",
        fontstyle="italic" if italic else "normal",
        transform=ax.transAxes, color="#333333",
    )


def draw_cell(ax, sample_index, color, edgecolor="white", linewidth=0.35):
    cell_w = DATA_FRAC / N
    x0 = LABEL_FRAC + sample_index * cell_w
    ax.add_patch(plt.Rectangle(
        (x0, 0.06), cell_w, 0.88, facecolor=color or C_OFF,
        edgecolor=edgecolor, linewidth=linewidth,
        transform=ax.transAxes, clip_on=False,
    ))


def render_data_row(ax, row_label, colors, italic=False):
    hide(ax)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    render_row_label(ax, row_label, italic=italic)
    for sample_index, color in enumerate(colors):
        draw_cell(ax, sample_index, color)


def render_ec_gene_row(ax, gene):
    """Draw one oncogene row; color alone identifies the amplicon ID."""
    hide(ax)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    render_row_label(ax, gene, italic=True)
    cell_w = DATA_FRAC / N

    for sample_index, sample in enumerate(samples):
        x0 = LABEL_FRAC + sample_index * cell_w
        assignment = sample_gene_amplicons[sample].get(gene, "absent")
        if assignment == "absent":
            draw_cell(ax, sample_index, C_OFF)
            continue

        if assignment is None:
            segments = [(None, C_EC_UNRESOLVED)]
        else:
            segments = [(amp_id, amp_color(amp_id)) for amp_id in assignment]

        segment_w = cell_w / len(segments)
        for segment_index, (amp_id, color) in enumerate(segments):
            segment_x = x0 + segment_index * segment_w
            ax.add_patch(plt.Rectangle(
                (segment_x, 0.06), segment_w, 0.88,
                facecolor=color, edgecolor="white", linewidth=0.35,
                transform=ax.transAxes, clip_on=False,
            ))

        # A dark outer border keeps split cells visually grouped.
        ax.add_patch(plt.Rectangle(
            (x0, 0.06), cell_w, 0.88, facecolor="none",
            edgecolor="#FFFFFF", linewidth=0.45,
            transform=ax.transAxes, clip_on=False,
        ))


def grad_colors(key, cmap_name, zero_off=False):
    values = pd.to_numeric(df[key], errors="coerce")
    vmax = float(values.max()) if values.notna().any() else 0.0
    norm = Normalize(vmin=0, vmax=vmax if vmax > 0 else 1.0)
    cmap = plt.get_cmap(cmap_name)
    colors = []
    for sample in samples:
        value = pd.to_numeric(df.loc[sample, key], errors="coerce")
        if pd.isna(value) or (zero_off and value == 0):
            colors.append(C_OFF)
        else:
            colors.append(matplotlib.colors.to_hex(cmap(norm(value))))
    return colors, vmax


# ── RENDER ALL ROWS ───────────────────────────────────────────────────────────
cbar_queue = {}
axes = [fig.add_subplot(gs[index]) for index in range(nrows)]

for ax_index, (row, ax) in enumerate(zip(rows_spec, axes[:-1])):
    if row[0] == "title":
        render_title(ax, row[2])
        continue

    _, _, row_label, key, row_type = row
    italic = row_type in ("ec_gene", "mutation")

    if row_type == "ec_gene":
        render_ec_gene_row(ax, key)
        continue
    if row_type == "binary_amp":
        colors = [C_AMP if df.loc[s, "MYCN_status"] == 1 else C_WT for s in samples]
    elif row_type == "binary_blue":
        colors = [C_CHR if df.loc[s, key] == 1 else C_OFF for s in samples]
    elif row_type == "grad_purple":
        colors, vmax = grad_colors(key, "Purples", zero_off=True)
        cbar_queue["ecDNA"] = (ax_index, "Purples", 0, vmax, "ecDNA\ncount")
    elif row_type == "grad_ylorrd":
        colors, vmax = grad_colors(key, "OrRd")
        cbar_queue["MYCN_CN"] = (ax_index, "OrRd", 0, vmax, "MYCN copy\nnumber")
    elif row_type == "grad_mycnfpkm":
        colors, vmax = grad_colors(key, "RdPu")
        cbar_queue["MYCN_FPKM"] = (ax_index, "RdPu", 0, vmax, "MYCN\nFPKM")
    elif row_type == "grad_chek1":
        colors, vmax = grad_colors(key, "Greens")
        cbar_queue["CHEK1"] = (ax_index, "Greens", 0, vmax, "CHEK1\nFPKM")
    elif row_type == "mutation":
        colors = [C_MUT if df.loc[s, key] == 1 else C_OFF for s in samples]
    else:
        colors = [C_OFF] * N
    render_data_row(ax, row_label, colors, italic=italic)


# ── SAMPLE LABELS ──────────────────────────────────────────────────────────────
ax_x = axes[-1]
hide(ax_x)
ax_x.set_xlim(0, 1)
ax_x.set_ylim(0, 1)
cell_w = DATA_FRAC / N
for sample_index, sample in enumerate(samples):
    center_x = LABEL_FRAC + (sample_index + 0.5) * cell_w
    ax_x.text(
        center_x, 0.98, sample, fontsize=FONT_SAMPLE, fontstyle="italic",
        rotation=45, ha="right", va="top",
        transform=ax_x.transAxes, color="#222222",
    )
ax_x.axhline(
    0.985, color="#999999", lw=0.6,
    xmin=LABEL_FRAC, xmax=LABEL_FRAC + DATA_FRAC,
)


# ── COLORBARS ─────────────────────────────────────────────────────────────────
fig.canvas.draw()
n_cbars = len(cbar_queue)
CBAR_H = 0.060
CBAR_GAP = 0.022
total_cb_h = n_cbars * CBAR_H + max(0, n_cbars - 1) * CBAR_GAP
plot_top = axes[0].get_position().y1
plot_bottom = axes[-1].get_position().y0
stack_y0 = (plot_top + plot_bottom - total_cb_h) / 2

for colorbar_index, colorbar_id in enumerate(reversed([
    "ecDNA", "MYCN_CN", "MYCN_FPKM", "CHEK1",
])):
    if colorbar_id not in cbar_queue:
        continue
    _, cmap_name, vmin, vmax, label = cbar_queue[colorbar_id]
    cb_bottom = stack_y0 + colorbar_index * (CBAR_H + CBAR_GAP)
    cax = fig.add_axes([CBAR_X, cb_bottom, CBAR_FW, CBAR_H])
    display_vmax = vmax if vmax > 0 else 1.0
    scalar = plt.cm.ScalarMappable(
        cmap=plt.get_cmap(cmap_name), norm=Normalize(vmin, display_vmax)
    )
    scalar.set_array([])
    colorbar = fig.colorbar(scalar, cax=cax, orientation="vertical")
    colorbar.set_ticks([vmin, display_vmax])
    colorbar.set_ticklabels([f"{vmin:.0f}", f"{vmax:.0f}"])
    colorbar.ax.tick_params(labelsize=FONT_CBAR, length=2.5, pad=2)
    colorbar.set_label(label, fontsize=FONT_CBAR, labelpad=4, linespacing=1.3)
    for spine in cax.spines.values():
        spine.set_linewidth(0.5)


# ── LEGENDS ───────────────────────────────────────────────────────────────────
fig.canvas.draw()
LEGEND_RIGHT_X = 0.985
LEGEND_BOTTOM_Y = 0.08
LEGEND_GAP = 0.006

status_items = [
    mpatches.Patch(facecolor=C_AMP, edgecolor="#888888", linewidth=0.5, label="MYCN amplified"),
    mpatches.Patch(facecolor=C_WT, edgecolor="#888888", linewidth=0.5, label="MYCN wild-type"),
    mpatches.Patch(facecolor=C_CHR, edgecolor="#888888", linewidth=0.5, label="Chromothripsis"),
    mpatches.Patch(facecolor=C_OFF, edgecolor="#888888", linewidth=0.5, label="Absent / WT"),
]
if has_mutations:
    status_items.append(
        mpatches.Patch(facecolor=C_MUT, edgecolor="#888888", linewidth=0.5, label="Mutated")
    )

status_legend = fig.legend(
    handles=status_items, ncol=len(status_items),
    fontsize=FONT_LEGEND, frameon=True, fancybox=False,
    edgecolor="#CCCCCC", framealpha=1,
    loc="lower right",
    bbox_to_anchor=(LEGEND_RIGHT_X, LEGEND_BOTTOM_Y),
    bbox_transform=fig.transFigure, borderpad=0.5,
    handlelength=1.0, handletextpad=0.4,
    columnspacing=0.8, labelspacing=0.25,
)
status_legend.get_frame().set_linewidth(0.5)

# Use the rendered status-legend height to place the amplicon legend directly
# above it. This keeps the gap small while guaranteeing no overlap.
fig.canvas.draw()
status_box = status_legend.get_window_extent(fig.canvas.get_renderer())
status_box_fig = status_box.transformed(fig.transFigure.inverted())
amplicon_legend_y = status_box_fig.y1 + LEGEND_GAP

amplicon_items = [
    mpatches.Patch(
        facecolor=amp_color(amp_id), edgecolor="#888888", linewidth=0.5,
        label=str(amp_id),
    )
    for amp_id in sorted(used_amp_ids)
]
if any(
    assignment is None
    for assignments in sample_gene_amplicons.values()
    for assignment in assignments.values()
):
    amplicon_items.append(
        mpatches.Patch(
            facecolor=C_EC_UNRESOLVED, edgecolor="#888888", linewidth=0.5,
            label="? (unresolved)",
        )
    )

if amplicon_items:
    amplicon_legend = fig.legend(
        handles=amplicon_items, title="ecDNA amplicon ID (within sample)",
        loc="lower right", ncol=min(len(amplicon_items), 10),
        fontsize=FONT_LEGEND, title_fontsize=FONT_LEGEND,
        frameon=True, fancybox=False, edgecolor="#CCCCCC", framealpha=1,
        bbox_to_anchor=(LEGEND_RIGHT_X, amplicon_legend_y),
        bbox_transform=fig.transFigure, borderpad=0.5,
        handlelength=1.0, handletextpad=0.35,
        columnspacing=0.7, labelspacing=0.25,
    )
    amplicon_legend.get_frame().set_linewidth(0.5)


# ── SAVE ──────────────────────────────────────────────────────────────────────
for extension in ("pdf", "png"):
    output_path = OUT_PREFIX.with_suffix(f".{extension}")
    fig.savefig(
        output_path, dpi=OUTPUT_DPI, facecolor="white", edgecolor="none",
        metadata={"Creator": "Matplotlib", "Title": "Neuroblastoma oncoplot"}
        if extension == "pdf" else None,
    )
    print(f"Saved -> {output_path} ({OUTPUT_DPI} dpi)")

plt.close(fig)
